In [ ]:
import mcstasscript as ms
import mcstastox as mx
import scipp as sc
from scipp.typing import VariableLike
import scipp as sc
import matplotlib.pyplot as plt
import numpy as np
from scippneutron.conversion.graph.beamline import beamline

import os


import sys 
current = os.getcwd()
sys.path.append(current + "/trex_data_reduction")
from trex_reduction import inelastic
from trex_reduction import produce_trex_event_object, v_bin_norm

In [ ]:
file_path = "/Users/bb24144/Documents/McStas/reduction_plet/mcstas_to_scipp/run_benzene/ISIS_LET_generated_1"

with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp_simple(source_name="SourceMantid",
                                            sample_name="iso_samp")
    
# Load event data into scipp 
event_object = scipp_object
# McStas provides absolute time, not time of flight
event_object.coords["tof"] = event_object.coords["t"]
# Add additional information required for inelastic scattering
event_object = produce_trex_event_object(event_object, file_path, "Monitor6")
event_object

In [ ]:
qens_graph = {**beamline(scatter=True), **inelastic}
sc.show_graph(qens_graph)

In [ ]:
event_object = event_object.transform_coords("dE", graph=qens_graph)
event_object = event_object.transform_coords("mag_Q", graph=qens_graph)
event_object = event_object.transform_coords("two_theta", graph=qens_graph)

event_object

# Vanad load and transform

In [ ]:
vanad_path = "/Users/bb24144/Documents/McStas/reduction_plet/mcstas_to_scipp/run_benzene/vanad_prod_2e11"


with mx.Read(vanad_path) as loaded_data:
    scipp_object = loaded_data.export_scipp_simple(source_name="SourceMantid",
                                            sample_name="iso_samp")
    
data = ms.load_data(vanad_path)

vanad_event = scipp_object
vanad_event.coords["tof"] = vanad_event.coords["t"]
vanad_event = produce_trex_event_object(vanad_event, file_path, "Monitor6")


vanad_event = vanad_event.transform_coords("dE", graph=qens_graph)
vanad_event = vanad_event.transform_coords("mag_Q", graph=qens_graph)
vanad_event = vanad_event.transform_coords("two_theta", graph=qens_graph)

vanad_event

In [ ]:
fig, ax = plt.subplots()
(event_object.hist(mag_Q=51)).plot(ax=ax, linestyle='-', color='tab:blue')
(vanad_event.hist(mag_Q=51)).plot(ax=ax, linestyle='--', color='tab:orange')
ax.axvline(np.pi/2)
# (event_object.hist(theta_test=51)).plot(ax=ax, linestyle='--', color='tab:orange')
plt.show()

In [ ]:
new_q = sc.linspace('mag_Q', 0.6,1.8,num=20, unit = '1/Angstrom')
new_omega = sc.linspace('dE', -2,2,num=300, unit = 'meV')

bin_list = [new_q,new_omega]

hist_event, hist_vanad, hist_norm = v_bin_norm(bin_list,event_object, vanad_event)

In [ ]:
fig, [ax1,ax2,ax3] = plt.subplots(1,3,figsize =(12,4))


hist_vanad.plot(ax=ax1)
hist_event.plot(ax=ax2)
hist_norm.plot(ax=ax3)

ax1.set_title('vanadium')
ax2.set_title('sample')
ax3.set_title('normalised sample')
plt.show()

In [ ]:
fig, ax = plt.subplots()

col = ['tab:blue', 'tab:orange', 'tab:red', 'tab:green']

for i in range(1,4,1):
    hist_norm['mag_Q', i].plot(ax=ax, color = col[i])
    ax.legend()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import sys
sys.path.append("/Users/bb24144/Documents/McStas/reduction_plet/plet")
from plet_data import PletData

plet_data = PletData('/Users/bb24144/Documents/McStas/reduction_plet/trex_data_reduction/plet_data/benzene_290_360_inc.nxspe', 3.60, omega_lims = [-4,4], q_lims = [0.4,1.8])

In [ ]:
new_q_plet = sc.linspace('q', 0.6,1.8,num=20, unit = '1/Angstrom')
new_omega_plet = sc.linspace('omega', -2,2,num=300, unit = 'meV')

plet_data.data = plet_data.data.rebin(q = new_q_plet, omega = new_omega_plet)

In [ ]:
hist_event

In [ ]:
plet_data.data.coords['q']

In [ ]:
fig, ax = plt.subplots()

summed = sc.nansum(plet_data.data, dim='q')
summed_no_var = sc.values(summed)  # drops variances, keeps values


(sc.nansum(hist_norm, dim = 'mag_Q') / sc.nansum(hist_norm, dim = 'mag_Q').max()).plot(ax = ax, linewidth=0.8)
(summed_no_var / summed_no_var.max()).plot(ax=ax, linewidth=0.8, color = 'tab:orange')
#ax.stairs(plet_data.data.nansum('q').values / (np.max(plet_data.data.nansum('q').values)), plet_data.data.coords['omega'].values, label = 'pLET-exp')
ax.legend()
plt.show()

In [ ]:
q_values = plet_data.data.coords['q'].values
n_q = hist_norm.sizes['mag_Q']  # use hist_norm's actual Q size, not plet's coords

ncols = 4
nrows = int(np.ceil(n_q / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.8),
                          sharex=True)
axes = axes.flatten()

for i in range(n_q):  # index-based, safe for both
    ax = axes[i]

    hist_slice = hist_norm['mag_Q', i]
    plet_slice = plet_data.data['q', i]
    plet_vals = plet_slice.values

    if i == 0:
        norm_scaler = sc.scalar(hist_slice.values.max())
        plet_scaler = np.nanmax(plet_vals)


    # --- hist_norm ---
    
    hist_norm1d = hist_slice / norm_scaler
    ax.stairs(hist_norm1d.values,
          hist_norm1d.coords['dE'].values,
          color='steelblue', linewidth=0.8, label='LET-McStas')

    # --- plet ---
    plet_norm = plet_vals / plet_scaler
    ax.stairs(plet_norm,
              plet_data.data.coords['omega'].values,
              color='tomato', linewidth=0.8, label='pLET-exp')

    q_val = q_values[i]

    ax.set_title(f'Q = {q_val:.2f} Å⁻¹', fontsize=10)
    ax.set_ylim(0,np.nanmax(hist_norm1d.values)+0.05)
    ax.set_ylabel('Normalised Intensity')
    ax.set_xlabel('ω (meV)')

# # Hide unused panels
# for j in range(n_q, len(axes)):
#     axes[j].set_visible(False)



axes[0].legend(loc='upper right', fontsize=10)

fig.suptitle('Experimental and Simulated LET benzene spectra', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
hist_norm

In [ ]:
plet_data.data

In [ ]:
q_mid = plet_data.data.coords['q'].values[0:-1] + np.diff(plet_data.data.coords['q'].values)/2 
q_mid

In [ ]:
plet_data.data.coords['omega']

In [ ]:
plet_int = np.zeros(19)
mc_int = np.zeros(19)   
vad_int = np.zeros(19)


for i in range(len(plet_data.data.coords['q']) - 1):
   plet_int[i] =  np.trapezoid(plet_data.data['q', i].values)
   mc_int[i] = np.trapezoid(hist_norm['mag_Q', i].values)
   vad_int[i] = np.trapezoid(hist_vanad['mag_Q', i].values)

In [ ]:
plt.plot(q_mid, plet_int / np.mean(plet_int))
plt.plot(q_mid, mc_int / np.mean(mc_int)) 

In [ ]:
def second_moment_analyser(energy, s_qw, s_qw_err = None):
    """
    Calculate the second moment of S(q,ω).

    Parameters|
    ----------
    energy: ndarray
         Array of energy values.
    s_qw: ndarray
        2D array where each row corresponds to the scattering function at a specific energy.

    Returns
    ----------
        second_moment: ndarray
            Array of second moment values as a function of q
    """
    s_qe = np.zeros(s_qw.shape[0])
    e2s_qe = np.zeros(s_qw.shape[0])

    if s_qw_err is not None:
        err_total = np.zeros(s_qw.shape[0])


    for i, sqw in enumerate(s_qw):
        s_qe[i] = np.trapezoid(y=sqw, x=energy)
        e2s_qe[i] = np.trapezoid(y=(energy**2 * sqw), x=energy)
        if s_qw_err is not None:
            _, s_qe_err = trap_uncertainty(x = energy, y = sqw, var_y = s_qw_err[i])
            _, e2s_qe_err = trap_uncertainty(x = energy, y = (energy**2 * sqw), var_y = (energy**2*s_qw_err[i]))
            err_total[i] = np.sqrt((e2s_qe[i] / s_qe[i]**2)**2 * s_qe_err**2 + (1/s_qe[i])**2 * e2s_qe_err**2)

    second_moment = e2s_qe / s_qe

    if s_qw_err is not None:
        return second_moment, err_total
    else:
        return second_moment
    

def trap_uncertainty(x, y, var_y):
    """
    Compute trapezoidal integral and propagated uncertainty for unevenly spaced data.
    Based off: 
    https://stats.stackexchange.com/questions/214850/propagate-errors-in-measured-points-to-simpsons-numerical-integral

    Parameters
    ----------
    x : array-like
        The x-values (must be in increasing order).
    y : array-like
        The corresponding y-values.
    var_y : array-like
        variances of y-values.

    Returns
    -------
    integral : float
        The computed trapezoidal integral.
    sigma_integral : float
        The propagated uncertainty in the integral.
    """
    N = len(x)
    h = np.diff(x)

    # Trapezoidal integral
    integral = np.sum(0.5 * h * (y[:-1] + y[1:]))

    # Error propagation
    sigma_I_sq = 0.0
    sigma_I_sq += (0.5 * h[0])**2 * var_y[0]
    sigma_I_sq += (0.5 * h[-1])**2 * var_y[-1]

    for j in range(1, N - 1):
        coeff = 0.5 * (h[j - 1] + h[j])
        sigma_I_sq += coeff**2 * var_y[j]

    sigma_integral = np.sqrt(sigma_I_sq)

    # Assert solution is similar to scipy implementation
    assert np.isclose(integral, np.trapezoid(x=x,y=y))

    return integral, sigma_integral



In [ ]:
omega_mid = plet_data.data.coords['omega'].values[0:-1] + np.diff(plet_data.data.coords['omega'].values)/2

In [ ]:
plet_2mom, plet_2mom_err = second_moment_analyser(energy = omega_mid, s_qw = plet_data.data.values, s_qw_err = plet_data.data.variances)
mc_2mom = second_moment_analyser(energy = omega_mid, s_qw = hist_norm.values)

In [ ]:
plt.errorbar(q_mid, plet_2mom, yerr = plet_2mom_err, label = 'pLET-exp', color = 'tab:orange')
plt.plot(q_mid, mc_2mom, label = 'LET-McStas', color = 'tab:blue')
plt.xlabel('Q (1/Å)')
plt.ylabel('Reduced second moment')
plt.legend()